In [1]:
import numpy as np
from numpy import cos, sin
import astropy as ast
from astropy.time import Time
import astropy.units as u
mu = 1.2150584270572e-2

In [2]:
ephem_file = open('Artemis_II_OEM_2026_04_09_Post-ICPS-Sep_to_EI.asc', 'r')
ephem_data = ephem_file.read()
ephem_rows = ephem_data.split("\n")[20:-1]

In [3]:
state_raw = np.zeros((6,np.shape(ephem_rows)[0]))
combined_state = np.zeros((6,np.shape(ephem_rows)[0]))
time_raw = [0]*len(ephem_rows)
time = np.zeros(np.shape(ephem_rows))
initial_time = Time(ephem_rows[0].split(" ")[0], format='isot', scale='utc').jd
for i in range(len(ephem_rows)):
    time_raw[i] = Time(ephem_rows[i].split(" ")[0], format='isot', scale='utc')
    time[i] = time_raw[i].jd - initial_time
    r_moon_gcrs = ast.coordinates.get_body('moon', time_raw[i])
    r_moon_earth = r_moon_gcrs.cartesian.xyz.to(u.km).value.T
    r_earth_emb = -mu*r_moon_earth
    r_moon_emb = (1 - mu)*r_moon_earth
    combined_state[:,i] = np.hstack((r_earth_emb, r_moon_emb))
    offset = np.hstack((r_earth_emb, np.zeros(3)))

    state_raw[:,i] = np.array(ephem_rows[i].split(" ")[1:7], dtype='float64') + offset

In [4]:
z_hat = np.cross(combined_state[3:6,0], combined_state[3:6,1])/np.linalg.norm(np.cross(combined_state[3:6,0], combined_state[3:6,1]))
x_hat = combined_state[3:6,0]/np.linalg.norm(combined_state[3:6,0])
y_hat = np.cross(z_hat, x_hat)
rot_mat = np.vstack((x_hat, y_hat, z_hat))

state = np.zeros(np.shape(state_raw))
for i in range(np.shape(state_raw)[1]):
    state[0:3,i] = rot_mat@state_raw[0:3,i]
    state[3:6,i] = rot_mat@state_raw[3:6,i]

In [5]:
np.savez(
        "free_return_inertial",
        state = state,
        time = time,
        combined_state = combined_state
    )

In [6]:
mu = 1.2150584270572e-2

def t_to_day(t):
    return t*27.321661/2/np.pi

def day_to_t(t):
    return t*2*np.pi/27.321661

def dist_to_x(x):
    return x/384400

def unit_convert(x):
    x_new = np.zeros(np.shape(x))
    for i in range(np.shape(x)[1]):
        x_new[0:3,i] = dist_to_x(x[0:3,i])
        x_new[3:6,i] = t_to_day(dist_to_x(x[3:6,i])*86400)
    return x_new

def unrotate_state(state, time_ang):
    state_rotating = np.zeros(np.shape(state))
    for i in range(np.shape(state)[1]):
        x, y, z, vx, vy, vz = state[:,i]
        theta = time_ang[i]
        state_rotating[:,i] = np.array([
            x*cos(theta) + y*sin(theta),
            y*cos(theta) - x*sin(theta),
            z,
            vx*cos(theta) + vy*sin(theta) + y*cos(theta) - x*sin(theta),
            vy*cos(theta) - vx*sin(theta) - x*cos(theta) - y*sin(theta),
            vz])
    return state_rotating
        

In [7]:
time_rotating = day_to_t(time)
state_rotating = unrotate_state(unit_convert(state), time_rotating)

In [8]:
np.savez(
    "free_return_rotating",
    state = state_rotating,
    time = time_rotating,
)